# Telecom-T2C — Inference Server on Kaggle (LoRA adapter -> ngrok tunnel)

A Kaggle-hosted alternative to `Telecom_T2C_Inference_Server.ipynb` (which
targets Colab) — same idea: load the trained LoRA adapter (never merged),
serve it over HTTP, tunnel it out via [ngrok](https://ngrok.com) so you can
call it from your own PC. Use this if Colab quota/availability is the
blocker, or you'd simply rather run this on Kaggle.

**What's different from the Colab notebook:**
- No Google Drive here — Kaggle has no equivalent mount. The adapter comes
  from a **Kaggle Dataset** you attach to this notebook instead (Section 4
  below has the exact steps).
- `utils.resolve_secret()` already checks Kaggle Secrets as well as Colab's
  (and a plain environment variable) — the ngrok-authtoken cell needs no
  Kaggle-specific change.
- The Local Smoke Test uses a hand-written example matching the training
  data's exact system-prompt/deployment-context shape, instead of pulling a
  real row from the validation set (that file lives on Drive, not reachable
  from here). For a smoke test against real validation data, use the Colab
  notebook.

**Before you start:** in this notebook's settings (right sidebar), turn on
**Internet** (needed for `pip install`, cloning this repo, and ngrok) and
select a **GPU** (T4 x2 or P100 — either has enough VRAM for a 4-bit 12B
model, unlike a typical laptop GPU).

Run cells top to bottom. Sections: Sync Code, Runtime Check, Install,
Configuration, Locate Adapter (Kaggle Dataset), Load Model, Local Smoke
Test, Start Server, Connect From Your PC, Stop Server.

## 0. Sync Code

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/sandeep-gupta-azalio/Telecom-T2C.git"
REPO_DIR = "/kaggle/working/Telecom-T2C"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo already present at {REPO_DIR} -- pulling latest changes...")
    status = subprocess.run(
        ["git", "-C", REPO_DIR, "status", "--porcelain"], capture_output=True, text=True
    ).stdout
    stashed = bool(status.strip())
    if stashed:
        print("Local changes detected -- stashing before pull.")
        subprocess.check_call(["git", "-C", REPO_DIR, "stash", "--include-untracked"])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])
    if stashed:
        try:
            subprocess.check_call(["git", "-C", REPO_DIR, "stash", "pop"])
        except subprocess.CalledProcessError:
            print(
                "WARNING: could not automatically restore your local changes (merge conflict "
                "with the pulled update) -- run `!git -C /kaggle/working/Telecom-T2C stash show -p` "
                "in a new cell to recover them manually, then resolve and `git stash drop`."
            )
else:
    print(f"Cloning {REPO_URL} into {REPO_DIR}...")
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print(f"cwd: {os.getcwd()}")


## 1. Runtime Check

In [ ]:
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Locate the Telecom-T2C project root from any Colab/local starting cwd."""
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for child in start.glob("*/"):
        if (child / "src").is_dir() and (child / "configs").is_dir():
            return child
    raise RuntimeError(
        "Could not locate the Telecom-T2C project root (a directory containing both "
        "'src/' and 'configs/'). If running in Colab, cd into the cloned/uploaded repo "
        "directory first, e.g.:\n  %cd /content/Telecom-T2C"
    )


PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
from src import utils

logger = utils.setup_logging()
gpu = utils.detect_gpu()
print(f"GPU: {gpu.name} | family={gpu.family} | vram={gpu.vram_gb:.1f} GB | bf16={gpu.bf16_supported}")

import torch

print(f"torch: {torch.__version__} | built for CUDA {torch.version.cuda}")


## 2. Install

Same phased install as the Colab notebooks — see
`Telecom_T2C_Trainer_v2.ipynb` Section 2 / `requirements.txt`'s top comment
for the full reasoning. Kaggle's base image differs from Colab's; if this
phased install hits an issue Colab never did, that's new information worth
capturing, not necessarily a bug in this notebook.

In [ ]:
import re
import subprocess
import sys

import torch

from src import utils

# Phase 1: everything that resolves normally (no --no-deps needed).
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "-r", str(PROJECT_ROOT / "requirements.txt"),
    ]
)

# Phase 2: the correlated Unsloth ML stack, installed together with
# --no-deps — see Telecom_T2C_Trainer_v2.ipynb Section 2 for the full
# reasoning behind this split.
torch_version = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
xformers_pin = "xformers==" + {
    "2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2",
}.get(torch_version, "0.0.34")
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "unsloth_zoo", "bitsandbytes>=0.46.1,!=0.48.0", "accelerate>=1.8",
        xformers_pin, "peft>=0.19.1", "trl>=0.15.0", "triton", "unsloth",
    ]
)

# Phase 3: torchao, also --no-deps.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "torchao>=0.16.0",
    ]
)

# Phase 4: transformers + tokenizers, --no-deps, installed last.
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--upgrade",
        "transformers==5.10.2", "tokenizers>=0.22.0,<=0.23.0",
    ]
)

# torchaudio: confirmed broken import chain on some Colab images, not
# needed by this text-only project — see Telecom_T2C_Trainer_v2.ipynb
# Section 2 for the full incident writeup.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"],
    check=False,
)

utils.disable_unused_transformers_backends()

# Serving-only dependencies (not needed for training, so not in requirements.txt).
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "fastapi", "uvicorn", "pyngrok"]
)

print("Install complete. If this is the first install in a fresh runtime, "
      "Runtime -> Restart session, then re-run from Section 1.")


## 3. Configuration

Loads `configs/experiment.yaml` for `model.base_model` /
`data.max_seq_length` / `model.hf_token_env_var` — the same config the
training run used. The dataset/Drive-related fields in this file are
irrelevant here (no dataset gets loaded in this notebook).

In [ ]:
from src import config as config_mod

CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment.yaml"
experiment_config = config_mod.load_config(CONFIG_PATH)
print(f"base_model: {experiment_config.model.base_model}")
print(f"max_seq_length: {experiment_config.data.max_seq_length}")


## 4. Locate Adapter (Kaggle Dataset)

Kaggle has no Google Drive mount, so the adapter has to arrive as a
**Kaggle Dataset** instead of being auto-detected from Drive:

1. Download the adapter from Google Drive to your own machine — either the
   raw `adapter/` folder, or the `.zip` the trainer notebook's Section 11
   (Save) already produces at
   `<google_drive_directory>/<run_name>/<experiment_name>.zip`.
2. On kaggle.com: **Datasets -> New Dataset**, upload the folder or zip.
   This creates a private dataset (e.g. `your-username/telecom-t2c-adapter`).
3. In this notebook's right sidebar: **+ Add Input**, search for that
   dataset, attach it. Its files appear read-only under
   `/kaggle/input/<dataset-slug>/...`.
4. Set `ADAPTER_DIR` or `ADAPTER_ZIP_PATH` below to match what you
   uploaded.

In [ ]:
import shutil
from pathlib import Path

# Set exactly ONE of these to match what you uploaded in step 2 above.
ADAPTER_DIR = "/kaggle/input/telecom-t2c-adapter/adapter"  # if you uploaded the raw adapter/ folder
ADAPTER_ZIP_PATH = None  # e.g. "/kaggle/input/telecom-t2c-adapter/telecom_t2c_gemma4.zip"

if ADAPTER_ZIP_PATH:
    # Kaggle dataset inputs are read-only, so a zip has to be extracted
    # into the writable /kaggle/working/ area first.
    extract_dir = Path("/kaggle/working/adapter")
    shutil.unpack_archive(ADAPTER_ZIP_PATH, extract_dir)
    adapter_dir = extract_dir
else:
    adapter_dir = Path(ADAPTER_DIR)

if not adapter_dir.is_dir():
    raise RuntimeError(
        f"{adapter_dir} not found. Attach your adapter as a Kaggle Dataset (notebook sidebar "
        "-> + Add Input) and set ADAPTER_DIR or ADAPTER_ZIP_PATH above to match its actual path."
    )
print(f"Using adapter: {adapter_dir}")


## 5. Load Model + Adapter

In [ ]:
from src import inference as inference_mod
from src import tokenizer as tokenizer_mod

hf_token = tokenizer_mod.resolve_hf_token(experiment_config.model.hf_token_env_var)
inf_model, inf_tokenizer = inference_mod.load_model_for_inference(
    experiment_config.model, experiment_config.data.max_seq_length, str(adapter_dir), hf_token,
)


## 6. Local Smoke Test

Hand-written example matching the training data's exact shape (system
prompt + `## Deployment context` + `## Query`) — not pulled from real
validation data, since that file lives on Drive and isn't reachable from a
Kaggle kernel. For a smoke test against a real validation row, use the
Colab inference-server notebook instead.

In [ ]:
SAMPLE_SYSTEM_PROMPT = (
    "You are a GPON network inventory query compiler. Given deployment context and "
    "natural language queries, emit five passes per query:\n"
    "PASS_0 Normalization \u2014 spelling/token fixes only; (none) when query is already clean.\n"
    "PASS_1 Lexical Detection \u2014 quoted verbatim phrases from normalized text (lexer output).\n"
    "PASS_2 Intent \u2014 exactly one canonical operation (LOOKUP, LIST, TRACE, COUNT, "
    "UNSUPPORTED on failure traces, etc.).\n"
    "PASS_3 Semantic Resolution \u2014 YAML semantic record only (mention, entity, source, confidence).\n"
    "PASS_4 TIR envelope JSON with status (SUCCESS or failure status) and diagnostics when not SUCCESS.\n"
    "Never invent identifiers or filters. Use deployment aliases only when listed in context."
)
SAMPLE_DEPLOYMENT_CONTEXT = (
    "## Deployment context\n\n"
    "product_families:\n"
    "  OLT:\n"
    "    aliases:\n"
    "      - OLT\n"
    "      - MA5xxx\n"
    "      - DSLAM\n"
)

prompt_messages = [
    {"role": "system", "content": SAMPLE_SYSTEM_PROMPT},
    {"role": "user", "content": SAMPLE_DEPLOYMENT_CONTEXT},
    {"role": "user", "content": "## Query\nList all OLT devices in the network."},
]
result = inference_mod.generate(inf_model, inf_tokenizer, prompt_messages, max_new_tokens=400)
print(result[:1500])


## 7. Start Inference Server + ngrok Tunnel

In [ ]:
from src import server as server_mod
from src import utils

ngrok_authtoken = utils.resolve_secret("NGROK_AUTHTOKEN")
if not ngrok_authtoken:
    from getpass import getpass

    ngrok_authtoken = getpass("Paste your ngrok authtoken (https://dashboard.ngrok.com/get-started/your-authtoken): ")

api_token = server_mod.generate_api_token()
app = server_mod.build_app(inf_model, inf_tokenizer, api_token, default_max_new_tokens=experiment_config.evaluation.max_new_tokens_eval)
sft_server, tunnel = server_mod.start_server(app, port=8000, ngrok_authtoken=ngrok_authtoken)

print(f"Public URL:   {tunnel.public_url}")
print(f"Bearer token: {api_token}")
print()
print("Example curl, using the SAME prompt_messages from Section 6")
print("(a bare one-line question with no system prompt will just get a")
print("generic-assistant reply, not PASS_0-4 output — see Section 6's note):")
import json as _json

_curl_body = _json.dumps({"messages": prompt_messages})
print(
    f"curl -X POST {tunnel.public_url}/generate "
    f"-H 'Authorization: Bearer {api_token}' -H 'Content-Type: application/json' "
    f"-d '{_curl_body}'"
)


## 8. Connect From Your Local PC

From your own machine (no GPU/Colab needed), send requests with the public
URL and bearer token printed above. **`messages` must include the same
system prompt + "Deployment context" turn the adapter was trained on** —
see Section 6's `prompt_messages` for a real, working example (copy its
printed JSON), or README "Dataset format" for the exact shape. A bare
`{"role": "user", "content": "<question>"}` with nothing else will not
produce PASS_0-4 output — the model will just answer like a generic
assistant, since that shape never appeared in training.

```python
import requests

BASE_URL = "https://<your-ngrok-subdomain>.ngrok-free.app"  # from Section 7's output
API_TOKEN = "<paste bearer token from Section 7>"

# Replace this with the real prompt_messages printed by Section 6 —
# system prompt + deployment-context turn + your actual query.
messages = [
    {"role": "system", "content": "You are a GPON network inventory query compiler..."},
    {"role": "user", "content": "## Deployment context\n\n..."},
    {"role": "user", "content": "## Query\nList all OLT devices in the network."},
]

response = requests.post(
    f"{BASE_URL}/generate",
    headers={"Authorization": f"Bearer {API_TOKEN}"},
    json={"messages": messages},
    timeout=120,
)
response.raise_for_status()
print(response.json()["generated_text"])
```

`GET {BASE_URL}/health` (no auth required) is a quick way to check the
tunnel is up before sending a real request.

## 9. Stop Server

In [ ]:
server_mod.stop_server(sft_server, tunnel)
print("Server stopped.")
